In [0]:
# Databricks notebook source
# ══════════════════════════════════════
# 00_UTILS — Funcoes Utilitarias
# Squad 3 — Arquitetura Medalhao
# Batch Lojas Fisicas
# ══════════════════════════════════════


In [0]:
%run ../config/00_config

In [0]:
# Importando bibliotecas

from pyspark.sql.functions import (
    col, lit, current_timestamp,
    to_timestamp, year, month,
    count, when, sum as spark_sum,
    trim, upper, lower
)
from pyspark.sql.types import StringType

print("Imports carregados!")

In [0]:
# Definicao de variaveis

# OAuth configuracoes para acesso ao ADLS

def build_adls_options(
    storage_account_name,
    client_id,
    tenant_id,
    client_secret
):
    """
    Monta dicionario de opcoes OAuth para
    autenticacao no ADLS via Service Principal.
    Recebe parametros explicitamente.
    """
    return {
        f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net"         : "OAuth",
        f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
        f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net"  : client_id,
        f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net": client_secret,
        f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token",
    }


def get_adls_options():
    """Retorna opcoes ADLS usando variaveis do 00_config."""
    return build_adls_options(
        ADLS_STORAGE_ACCOUNT_NAME,
        ADLS_CLIENT_ID,
        ADLS_TENANT_ID,
        ADLS_CLIENT_SECRET
    )

print("Funcoes de autenticacao ADLS criadas!")

In [0]:
# leitura CSV via Spark

def read_source_csv(spark, source_path, adls_options, csv_options=None):
    """
    Le arquivo CSV do ADLS via Spark com autenticacao OAuth.
    Recebe spark como parametro — sem dependencia de variavel global.
    Parametros:
        spark        : SparkSession
        source_path  : caminho completo abfss://
        adls_options : dict de opcoes OAuth
        csv_options  : dict de opcoes CSV (opcional)
    """
    csv_options = csv_options or {}

    return (
        spark.read
        .format("csv")
        .options(**csv_options)
        .options(**adls_options)
        .load(source_path)
    )

print("Funcao read_source_csv criada!")

In [0]:
# DBTITLE 1,Validacoes de qualidade de dados

# validacoes de qualidade de dados

def validate_required_columns(df, expected_columns):
    """
    Valida se o DataFrame contem todas as colunas esperadas.
    Lanca erro explicito se colunas obrigatorias estiverem ausentes.
    """
    if not expected_columns:
        return {
            "validation_applied" : False,
            "missing_columns"    : [],
            "unexpected_columns" : df.columns,
            "message"            : "EXPECTED_COLUMNS nao informado. Validacao ignorada.",
        }

    missing_columns    = [c for c in expected_columns if c not in df.columns]
    unexpected_columns = [c for c in df.columns if c not in expected_columns]

    if missing_columns:
        raise ValueError(
            f"Colunas obrigatorias ausentes: {missing_columns}"
        )

    return {
        "validation_applied" : True,
        "missing_columns"    : missing_columns,
        "unexpected_columns" : unexpected_columns,
        "message"            : "Validacao de colunas concluida com sucesso.",
    }


def validate_key_columns(df, key_columns):
    """
    Valida se colunas candidatas a chave primaria (PK)
    nao possuem nulos nem duplicatas.
    Lanca erro explicito com detalhes se falhar.
    """
    if not key_columns:
        return {
            "validation_applied" : False,
            "key_columns"        : [],
            "null_counts"        : {},
            "duplicated_rows"    : None,
            "message"            : "KEY_COLUMNS nao informado. Validacao ignorada.",
        }

    missing_key_columns = [
        c for c in key_columns
        if c not in df.columns
    ]

    if missing_key_columns:
        raise ValueError(
            f"Colunas de chave ausentes no DataFrame: {missing_key_columns}"
        )

    null_counts = {}
    for column_name in key_columns:
        null_count = df.filter(col(column_name).isNull()).count()
        null_counts[column_name] = null_count

        if null_count > 0:
            raise ValueError(
                f"Coluna de chave '{column_name}' possui "
                f"{null_count} registros nulos."
            )

    total_rows        = df.count()
    distinct_key_rows = df.select(*key_columns).distinct().count()
    duplicated_rows   = total_rows - distinct_key_rows

    if duplicated_rows > 0:
        raise ValueError(
            f"Encontrados {duplicated_rows} registros duplicados "
            f"na chave {key_columns}. "
            f"Total={total_rows} | Distintos={distinct_key_rows}."
        )

    return {
        "validation_applied" : True,
        "key_columns"        : key_columns,
        "null_counts"        : null_counts,
        "duplicated_rows"    : duplicated_rows,
        "message"            : "Validacao de chave concluida com sucesso.",
    }


def validate_partition_date(df, date_column):
    """
    Valida se coluna de data pode ser usada para particionamento.
    Lanca erro se houver falhas de conversao para timestamp.
    """
    df_test = df.withColumn(
        "_dt_converted",
        to_timestamp(col(date_column))
    )

    df_validacao = df_test.select(
        count("*").alias("total_linhas"),
        count(when(col(date_column).isNull(), True)).alias("data_nula_origem"),
        count(
            when(
                col(date_column).isNotNull() &
                col("_dt_converted").isNull(),
                True
            )
        ).alias("falhas_conversao")
    )

    display(df_validacao)
    validacao = df_validacao.collect()[0]

    if validacao["falhas_conversao"] > 0:
        raise ValueError(
            f"Coluna '{date_column}' possui "
            f"{validacao['falhas_conversao']} registros que nao "
            f"podem ser convertidos para timestamp."
        )

    print(f"Validacao OK: '{date_column}' pode ser usada para particionamento.")
    return validacao

print("Funcoes de validacao criadas!")

In [0]:
# DBTITLE 1,validacoes de qualidade de dados

# funcoes de auditoria e metadados

def add_bronze_metadata(df, source_file):
    """
    Adiciona colunas de auditoria na camada Bronze.
    Usa current_timestamp() do Spark.
    """
    return (
        df
        .withColumn("bronze_ingested_at", current_timestamp())
        .withColumn("bronze_source_file", lit(source_file))
    )


def add_silver_metadata(df):
    """
    Adiciona coluna de auditoria na camada Silver.
    Usa current_timestamp() do Spark.
    """
    return df.withColumn("silver_processed_at", current_timestamp())


def add_partition_columns(df, date_column):
    """
    Adiciona colunas de particao ano e mes
    extraidas de uma coluna de data.
    """
    return (
        df
        .withColumn("ano", year(to_timestamp(col(date_column))))
        .withColumn("mes", month(to_timestamp(col(date_column))))
    )


def cast_all_columns_to_string(df):
    """
    Converte todas as colunas para StringType.
    Util para cargas Bronze — preserva dado proximo da origem.
    """
    for column_name in df.columns:
        df = df.withColumn(
            column_name,
            col(column_name).cast(StringType())
        )
    return df

print("Funcoes de auditoria e metadados criadas!")

In [0]:
# funcoes de escrita e leitura Delta no ADLS

def write_delta(df, path, mode="overwrite",
                partition_by=None, adls_options=None):
    """
    Grava Spark DataFrame como Delta no ADLS.
    Grava fisicamente no ADLS via abfss://.
    """
    adls_options = adls_options or get_adls_options()

    writer = (
        df.write
        .format("delta")
        .options(**adls_options)
        .option("overwriteSchema", "true")
        .mode(mode)
    )

    if partition_by:
        writer = writer.partitionBy(partition_by)

    writer.save(path)

    contagem = df.count()
    print(f"Delta gravado com sucesso!")
    print(f"   Caminho   : {path}")
    print(f"   Modo      : {mode}")
    print(f"   Particoes : {partition_by}")
    print(f"   Linhas    : {contagem:,}")


def read_delta(spark, path, adls_options=None):
    """
    Le tabela Delta do ADLS como Spark DataFrame.
    """
    adls_options = adls_options or get_adls_options()

    df = (
        spark.read
        .format("delta")
        .options(**adls_options)
        .load(path)
    )

    contagem = df.count()
    print(f"Delta lido com sucesso!")
    print(f"   Caminho : {path}")
    print(f"   Linhas  : {contagem:,}")
    return df


def delta_merge(spark, df_new, path,
                merge_condition, adls_options=None):
    """
    Realiza upsert (merge) em tabela Delta no ADLS.
    Atualiza registros existentes e insere novos.
    """
    from delta.tables import DeltaTable

    adls_options = adls_options or get_adls_options()

    delta_table = DeltaTable.forPath(spark, path)

    (
        delta_table.alias("target")
        .merge(df_new.alias("source"), merge_condition)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"Delta Merge executado com sucesso!")
    print(f"   Caminho   : {path}")
    print(f"   Condicao  : {merge_condition}")

print("Funcoes de escrita e leitura Delta criadas!")

In [0]:
# funcoes de escrita e leitura SQL Server

def write_sql_table(df, table_name, mode="overwrite"):
    """
    Grava Spark DataFrame no SQL Server.
    Todos os campos convertidos para STRING.
    """
    df_str = df.select([
        col(c).cast(StringType()).alias(c)
        for c in df.columns
    ])

    (
        df_str.write
        .format("sqlserver")
        .option("host", SQL_HOST)
        .option("port", SQL_PORT)
        .option("database", SQL_DATABASE)
        .option("dbtable", table_name)
        .option("user", SQL_USERNAME)
        .option("password", SQL_PASSWORD)
        .mode(mode)
        .save()
    )

    contagem = df.count()
    print(f"SQL Server gravado com sucesso!")
    print(f"   Tabela : {table_name}")
    print(f"   Linhas : {contagem:,}")
    print(f"   Modo   : {mode}")


def read_sql_table(spark, table_name):
    """
    Le tabela do SQL Server como Spark DataFrame.
    """
    df = (
        spark.read
        .format("sqlserver")
        .option("host", SQL_HOST)
        .option("port", SQL_PORT)
        .option("database", SQL_DATABASE)
        .option("dbtable", table_name)
        .option("user", SQL_USERNAME)
        .option("password", SQL_PASSWORD)
        .load()
    )

    contagem = df.count()
    print(f"SQL Server lido com sucesso!")
    print(f"   Tabela : {table_name}")
    print(f"   Linhas : {contagem:,}")
    return df

print("Funcoes SQL Server criadas!")

In [0]:
# funcoes de validacao de carga

def compare_row_counts(source_df, target_df, label=""):
    """
    Compara quantidade de linhas entre origem e destino.
    Lanca erro explicito com valores exatos se divergir.
    """
    source_count = source_df.count()
    target_count = target_df.count()

    print(f"Validacao de carga {label}:")
    print(f"   Linhas origem  : {source_count:,}")
    print(f"   Linhas destino : {target_count:,}")

    if source_count != target_count:
        raise ValueError(
            f"DIVERGENCIA DE CARGA {label}! "
            f"Origem={source_count:,} | "
            f"Destino={target_count:,} | "
            f"Diferenca={abs(source_count - target_count):,}"
        )

    print(f"   Status         : OK")
    return {
        "source_count" : source_count,
        "target_count" : target_count,
        "message"      : "Quantidade de registros validada com sucesso.",
    }


def validate_bronze_quality(df, has_partitions=True):
    """
    Valida qualidade minima da camada Bronze.
    Verifica colunas de auditoria obrigatorias.
    """
    select_cols = [
        count("*").alias("total_linhas"),
        count(when(col("bronze_ingested_at").isNull(), True)).alias("bronze_ingested_at_nulo"),
        count(when(col("bronze_source_file").isNull(), True)).alias("bronze_source_file_nulo"),
    ]

    if has_partitions:
        select_cols += [
            count(when(col("ano").isNull(), True)).alias("ano_nulo"),
            count(when(col("mes").isNull(), True)).alias("mes_nulo"),
        ]

    df_validacao = df.select(*select_cols)
    display(df_validacao)
    validacao = df_validacao.collect()[0]

    if validacao["bronze_ingested_at_nulo"] > 0:
        raise Exception("Bronze: existem registros sem bronze_ingested_at.")
    if validacao["bronze_source_file_nulo"] > 0:
        raise Exception("Bronze: existem registros sem bronze_source_file.")
    if has_partitions:
        if validacao["ano_nulo"] > 0:
            raise Exception("Bronze: existem registros com ano nulo.")
        if validacao["mes_nulo"] > 0:
            raise Exception("Bronze: existem registros com mes nulo.")

    print(f"Validacao qualidade Bronze OK: {validacao['total_linhas']:,} linhas.")
    return validacao


def validate_silver_quality(df, required_columns):
    """
    Valida qualidade minima da camada Silver.
    Verifica colunas obrigatorias e silver_processed_at.
    """
    missing = [c for c in required_columns if c not in df.columns]
    if missing:
        raise ValueError(
            f"Silver: colunas obrigatorias ausentes: {missing}"
        )

    null_processed = df.filter(
        col("silver_processed_at").isNull()
    ).count()

    if null_processed > 0:
        raise ValueError(
            f"Silver: {null_processed} registros sem silver_processed_at."
        )

    print(f"Validacao qualidade Silver OK!")

print("Funcoes de validacao de carga criadas!")

In [0]:
# funcao para retornar um dicionario com informacoes basicas sobre um DataFrame

def get_dataframe_overview(df):
    """
    Retorna informacoes basicas sobre um DataFrame.
    """
    return {
        "total_linhas"  : df.count(),
        "total_colunas" : len(df.columns),
        "colunas"       : df.columns,
    }

print("Funcao get_dataframe_overview criada!")

In [0]:
# funcoes de leitura e escrita 

print("=" * 55)
print("00_utils carregado com sucesso!")
print("=" * 55)
print("""
Funcoes disponiveis:

Autenticacao ADLS:
   build_adls_options()         -> dict OAuth
   get_adls_options()           -> atalho

Leitura:
   read_source_csv()            -> CSV via Spark
   read_delta()                 -> Delta do ADLS
   read_sql_table()             -> SQL Server

Escrita:
   write_delta()                -> Delta no ADLS
   delta_merge()                -> upsert Delta
   write_sql_table()            -> SQL Server (string)

Auditoria:
   add_bronze_metadata()        -> bronze_ingested_at + source_file
   add_silver_metadata()        -> silver_processed_at
   add_partition_columns()      -> ano + mes
   cast_all_columns_to_string() -> converte para string

Validacao:
   validate_required_columns()  -> colunas obrigatorias
   validate_key_columns()       -> PK nulos + duplicatas
   validate_partition_date()    -> data para particao
   compare_row_counts()         -> origem vs destino
   validate_bronze_quality()    -> auditoria bronze
   validate_silver_quality()    -> auditoria silver

Analise:
   get_dataframe_overview()     -> visao geral
""")